# 04 — Visualization & Real-World Validation
Vibe Correlation Map (radar chart) showing audio DNA per genre, plus real-world song prediction.
With 114 genres the radar would be unreadable — we pick a representative subset of 15 for the chart,
but compute audio DNA for all genres so you can swap in any subset you want.

In [ ]:
# Cell 1 — Imports
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# Cell 2 — Load dataset and define all features
# We use ALL_FEATURES here to match exactly what the model was trained on
ALL_FEATURES = [
    'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
    'key', 'loudness', 'mode', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
]

df = pd.read_csv("../data/SpotifySongs.csv", index_col=0)
df = df[ALL_FEATURES + ['track_genre']].copy()
print(f"Loaded {len(df):,} tracks across {df['track_genre'].nunique()} genres")

In [ ]:
# Cell 3 — Normalize features to 0-1 scale and compute mean per genre
# The radar chart needs everything on the same scale.
# Features like loudness (dB, negative) and duration_ms (large numbers) need min-max scaling.
# Features already in 0-1 range still go through this — it's a no-op for them.
df_norm = df.copy()
for col in ALL_FEATURES:
    col_min = df[col].min()
    col_max = df[col].max()
    if col_max != col_min:  # avoid divide-by-zero for constant features
        df_norm[col] = (df[col] - col_min) / (col_max - col_min)

# Compute mean feature value per genre — this is the "audio DNA"
audio_dna = df_norm.groupby('track_genre')[ALL_FEATURES].mean()
print(f"Audio DNA computed for {len(audio_dna)} genres")
audio_dna.round(3).head(10)

In [ ]:
# Cell 4 — Vibe Correlation Map: radar/spider chart
# 114 genres on one chart would be a mess — we pick 15 diverse genres to show
# the shape of the feature space. Swap in any genres from audio_dna.index you want.
CHART_GENRES = [
    'hip-hop', 'pop', 'rock', 'classical', 'country',
    'electronic', 'r-n-b', 'jazz', 'metal', 'ambient',
    'latin', 'reggae', 'blues', 'folk', 'techno'
]

# Only keep genres that exist in the dataset
CHART_GENRES = [g for g in CHART_GENRES if g in audio_dna.index]

N = len(ALL_FEATURES)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

fig, ax = plt.subplots(figsize=(11, 11), subplot_kw=dict(polar=True))
colors = plt.cm.tab20(np.linspace(0, 1, len(CHART_GENRES)))

for genre, color in zip(CHART_GENRES, colors):
    values = audio_dna.loc[genre].tolist()
    values += values[:1]  # close the polygon
    ax.plot(angles, values, linewidth=2, label=genre, color=color)
    ax.fill(angles, values, alpha=0.06, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(ALL_FEATURES, fontsize=9)
ax.set_yticklabels([])
ax.set_title("Vibe Correlation Map — Audio DNA per Genre (15 of 114)", fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15), fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — Real-world validation: predict the genre of an unseen song
# Fill in a song's audio features and the model will classify it.
# Features you can look up at tunebat.com or via the Spotify Web API.
# NOTE: popularity, key, mode, time_signature, and duration_ms are also required
# now that the model trains on all 15 features.

rf     = joblib.load("../models/best_model.pkl")   # loads whichever model won the comparison
le     = joblib.load("../models/label_encoder.pkl")
scaler = joblib.load("../models/scaler.pkl")

song = {
    'name'            : 'Choosin - Texas / Ella Langley',
    'popularity'      : 82,       # 0–100
    'duration_ms'     : 187000,   # in milliseconds
    'explicit'        : 0,        # 0 = clean, 1 = explicit
    'danceability'    : 0.68,
    'energy'          : 0.78,
    'key'             : 5,        # 0–11 (F = 5)
    'loudness'        : -5.0,     # dB
    'mode'            : 1,        # 0 = minor, 1 = major
    'speechiness'     : 0.03,
    'acousticness'    : 0.01,
    'instrumentalness': 0.0,
    'liveness'        : 0.19,
    'valence'         : 0.49,
    'tempo'           : 112.0,
    'time_signature'  : 4,        # beats per measure
}

FEATURE_ORDER = [
    'popularity', 'duration_ms', 'explicit', 'danceability', 'energy',
    'key', 'loudness', 'mode', 'speechiness', 'acousticness',
    'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature',
]

features        = np.array([[song[f] for f in FEATURE_ORDER]])
features_scaled = scaler.transform(features)

pred_class = rf.predict(features_scaled)[0]
pred_genre = le.inverse_transform([pred_class])[0]
pred_proba = rf.predict_proba(features_scaled)[0]

print(f"Song            : {song['name']}")
print(f"Predicted genre : {pred_genre.upper()}")
print()
print("Top 5 probabilities:")
for idx in np.argsort(pred_proba)[::-1][:5]:
    print(f"  {le.classes_[idx]:<20} {pred_proba[idx]:.1%}")

In [ ]:
# Cell 6 — Live API prediction (optional)
# Fetches real audio features from the track-analysis API and runs a prediction.
# Requires an active RapidAPI key.
import requests

API_KEY  = "72779a9516mshdf850e8303ae898p18c18bjsn2078f1551645"
BASE_URL = "https://track-analysis.p.rapidapi.com/pktx/analysis"

song_name   = input("Enter song name: ")
artist_name = input("Enter artist name (optional, press Enter to skip): ").strip()

params = {"song": song_name}
if artist_name:
    params["artist"] = artist_name

headers = {
    "x-rapidapi-key" : API_KEY,
    "x-rapidapi-host": "track-analysis.p.rapidapi.com"
}

try:
    response = requests.get(BASE_URL, headers=headers, params=params)
    response.raise_for_status()
    data = response.json()
    print(f"\nRaw API response: {data}")

    # API doesn't return popularity, key, mode, time_signature, explicit, or duration_ms
    # so we default those to typical values — they'll have less impact on prediction
    features_live = {
        'popularity'      : 50,   # unknown — use neutral default
        'duration_ms'     : 210000,
        'explicit'        : 0,
        'danceability'    : data.get('danceability', 0)     / 100,
        'energy'          : data.get('energy', 0)           / 100,
        'key'             : 0,    # unknown from this API
        'loudness'        : float(str(data.get('loudness', '0')).replace('dB', '').strip()),
        'mode'            : 1,    # unknown — default major
        'speechiness'     : data.get('speechiness', 0)      / 100,
        'acousticness'    : data.get('acousticness', 0)     / 100,
        'instrumentalness': data.get('instrumentalness', 0) / 100,
        'liveness'        : data.get('liveness', 0)         / 100,
        'valence'         : data.get('happiness', 0)        / 100,
        'tempo'           : data.get('tempo', 0),
        'time_signature'  : 4,    # unknown — default 4/4
    }

    features_arr    = np.array([[features_live[f] for f in FEATURE_ORDER]])
    features_scaled = scaler.transform(features_arr)

    pred_class = rf.predict(features_scaled)[0]
    pred_genre = le.inverse_transform([pred_class])[0]
    pred_proba = rf.predict_proba(features_scaled)[0]

    print(f"\nSong            : {song_name}" + (f" — {artist_name}" if artist_name else ""))
    print(f"Predicted genre : {pred_genre.upper()}")
    print("-" * 30)
    print("Top 5 probabilities:")
    for idx in np.argsort(pred_proba)[::-1][:5]:
        print(f"  {le.classes_[idx]:<20} {pred_proba[idx]:.1%}")

except Exception as e:
    print(f"An error occurred: {e}")